In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from scipy.stats import gaussian_kde
from datetime import datetime
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
import calendar




In [19]:


sp500_top25 = {
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Amazon": "AMZN",
    "Meta": "META",
    "Alphabet Class A": "GOOGL",
    "Alphabet Class C": "GOOG",
    "Berkshire Hathaway": "BRK-B",
    "Eli Lilly": "LLY",
    "Broadcom": "AVGO",
    "Tesla": "TSLA",
    "JPMorgan": "JPM",
    "UnitedHealth": "UNH",
    "Visa": "V",
    "Exxon Mobil": "XOM",
    "Mastercard": "MA",
    "Johnson & Johnson": "JNJ",
    "Procter & Gamble": "PG",
    "Home Depot": "HD",
    "Costco": "COST",
    "Merck": "MRK",
    "Chevron": "CVX",
    "AbbVie": "ABBV",
    "PepsiCo": "PEP",
    "Walmart": "WMT"
}


input_folder = "SP500_Top25_tratados"
df_todas_rend_medio = pd.DataFrame()   # ← inicialización correcta

for name, ticker in sp500_top25.items():

    df_train = pd.read_parquet(f"{input_folder}/{name}_{ticker}_train.parquet")
    df_una_rend_medio = df_train[["rendimiento_medio"]].rename(
        columns={"rendimiento_medio": name}
    )

    if df_todas_rend_medio.empty:
        df_todas_rend_medio = df_una_rend_medio
    else:
        df_todas_rend_medio = df_todas_rend_medio.join(df_una_rend_medio, how="outer")

# display(df_todas_rend_medio)



# Matriz de correlación completa
corr = df_todas_rend_medio.corr(method="pearson")

# ORDENAR filas y columnas alfabéticamente
corr = corr.sort_index(axis=0).sort_index(axis=1)

# Crear máscara para dejar solo la triangular inferior
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# Aplicar máscara (parte superior → NaN)
corr_lower = corr.mask(mask)


fig = px.imshow(
    corr_lower,
    text_auto=True,
    zmin=-1,
    zmax=1,
    color_continuous_scale=[
        [0.0, "yellow"],     # -1
        [0.5, "white"],    #  0
        [1.0, "blue"]       # +1
    ],
    title="Matriz de correlación (pearson)"
)

fig.update_layout(width=1200, height=900, xaxis_title="", yaxis_title="")
fig.show()


# Matriz de correlación completa
corr = df_todas_rend_medio.corr(method="spearman")

# ORDENAR filas y columnas alfabéticamente
corr = corr.sort_index(axis=0).sort_index(axis=1)

# Crear máscara para dejar solo la triangular inferior
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

# Aplicar máscara (parte superior → NaN)
corr_lower = corr.mask(mask)


fig = px.imshow(
    corr_lower,
    text_auto=True,
    zmin=-1,
    zmax=1,
    color_continuous_scale=[
        [0.0, "yellow"],     # -1
        [0.5, "white"],    #  0
        [1.0, "blue"]       # +1
    ],
    title="Matriz de correlación (spearman)"
)

fig.update_layout(width=1200, height=900, xaxis_title="", yaxis_title="")
fig.show()

In [4]:
# Preparar las matrices
corr_pearson = df_todas_rend_medio.corr(method="pearson")
corr_spearman = df_todas_rend_medio.corr(method="spearman")


In [5]:
# Convertir ambas matrices en formato largo (pares de activos)

# Asegurar que no hay nombres de índice que causen conflictos
corr_pearson.index.name = None
corr_pearson.columns.name = None
corr_spearman.index.name = None
corr_spearman.columns.name = None

# Convertir matrices a formato largo
df_pairs = (
    corr_pearson.stack()
    .reset_index()
    .rename(columns={0: "pearson", "level_0": "asset1", "level_1": "asset2"})
)

df_pairs["spearman"] = corr_spearman.stack().values

# Eliminar diagonal
df_pairs = df_pairs[df_pairs["asset1"] != df_pairs["asset2"]]

# Diferencia
df_pairs["diff"] = df_pairs["spearman"] - df_pairs["pearson"]



In [6]:
# Quitamos la diagonal (correlación consigo mismo):
df_pairs = df_pairs[df_pairs["asset1"] != df_pairs["asset2"]]


In [7]:
# Calcular la diferencia entre métodos
df_pairs["diff"] = df_pairs["spearman"] - df_pairs["pearson"]


In [8]:
# Gráfico comparativo Pearson vs Spearman

fig = px.scatter(
    df_pairs,
    x="pearson",
    y="spearman",
    color="diff",
    hover_data=["asset1", "asset2"],
    color_continuous_scale="RdBu",
    title="Comparación Pearson vs Spearman entre activos",
    labels={
        "pearson": "Correlación Pearson",
        "spearman": "Correlación Spearman",
        "diff": "Diferencia Spearman - Pearson"
    }
)

# Línea diagonal (donde Pearson = Spearman)
fig.add_shape(
    type="line",
    x0=-1, y0=-1, x1=1, y1=1,
    line=dict(color="black", width=1, dash="dash")
)

fig.update_layout(width=800, height=700)
fig.show()


📌 Cómo interpretar el gráfico
Puntos cerca de la diagonal → Pearson ≈ Spearman → relación lineal.

Puntos por encima de la diagonal → Spearman > Pearson → relación monótona pero no lineal.

Puntos por debajo → Pearson > Spearman → relación lineal fuerte pero con rangos menos consistentes.

Color rojo intenso → gran diferencia entre métodos.

Esto te permite ver qué activos tienen relaciones más robustas (Spearman) o más lineales (Pearson)

In [ ]:
# Ranking de pares con mayor diferencia

df_pairs.sort_values("diff", key=abs, ascending=False).head(10)


,asset1,asset2,pearson,spearman,diff
264,Tesla,Exxon Mobil,-0.033365,0.073987,0.107353
360,Exxon Mobil,Tesla,-0.033365,0.073987,0.107353
535,Chevron,Tesla,0.226465,0.328610,0.102146
271,Tesla,Chevron,0.226465,0.328610,0.102146
310,UnitedHealth,Tesla,0.671379,0.772412,0.101033
262,Tesla,UnitedHealth,0.671379,0.772412,0.101033
385,Mastercard,Tesla,0.725397,0.819022,0.093625
265,Tesla,Mastercard,0.725397,0.819022,0.093625
561,AbbVie,JPMorgan,0.696366,0.602877,-0.093490
297,JPMorgan,AbbVie,0.696366,0.602877,-0.093490
